# 02 Cleaning Demo

Demo cleaning truc tiep tu Bronze va Silver tren HDFS (khong dung file report JSON).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('CleaningDemoNotebook')
    .master('local[*]')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

BASE = 'hdfs://namenode:9000/nyc_taxi'
BRONZE = f'{BASE}/bronze/raw/yellow_tripdata_2026-01.parquet'
SILVER = f'{BASE}/silver/cleaned/yellow_taxi_cleaned_2026_01'

print(BRONZE)
print(SILVER)

In [ ]:
df_bronze = spark.read.parquet(BRONZE)
df_silver = spark.read.parquet(SILVER)

bronze_count = df_bronze.count()
silver_count = df_silver.count()
removed = bronze_count - silver_count

print(f'Bronze rows : {bronze_count:,}')
print(f'Silver rows : {silver_count:,}')
print(f'Removed     : {removed:,} ({removed/bronze_count*100:.2f}%)')

In [ ]:
print('Silver schema with engineered features:')
df_silver.printSchema()

In [ ]:
summary = df_silver.select(
    F.round(F.avg('trip_distance'), 2).alias('avg_trip_distance'),
    F.round(F.avg('fare_amount'), 2).alias('avg_fare'),
    F.round(F.avg('trip_duration_min'), 2).alias('avg_trip_duration_min'),
    F.round(F.avg('tip_percentage'), 2).alias('avg_tip_pct'),
    F.round(F.avg('speed_mph'), 2).alias('avg_speed_mph')
)
summary.show(truncate=False)

In [ ]:
print('Top 10 busy pickup hours (from Silver):')
df_silver.groupBy('pickup_hour').count().orderBy(F.col('count').desc()).show(10)